# Imports

In [ ]:
import json
import os

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from uavloc.data.drone_frames import DroneFrame, load_frames
from uavloc.data.multidate import load_date
from uavloc.datasets import dataset_ref
from uavloc.features.factory import correlator_config
from uavloc.features.segmentation import SegmentationFeatures, load_segmenter
from uavloc.matching.benchmark import benchmark_draws, score_draws, summarise
from uavloc.matching.identity import SatelliteAsDroneCorrelator
from uavloc.track import REPO_ROOT, set_seed

# Params

In [ ]:
BUILD = "corrector-identity-bench"

WINDOW = "rect"                                      # correlator setup of 006.1 / 007.1 / 021
SEARCH_SCALE = 1.7
PLACES = ["bushcha-50p2422-25p8305", "birky-49p9105-23p9192"]
MAP_DATE = "newest"
MATCHING = "weighted-ncc"
CLASS_WEIGHTS = (0.0, 1.0, 1.0, 0.5, 0.0, 0.5)       # background, building, track, woody, field, water
NOISE_M = [5.0, 10.0, 15.0]
PRIMARY_NOISE_M = 10.0
HARM_MARGIN_M = 2.0
N_DIRECTIONS = 8
SEED = 42
DEVICE = "mps"
FOV_W_DEG = 60.0
SAT_FEATURE_CROP_PCT = 0.03
DRONE_FEATURE_CROP_PCT = 0.0
SEG_ARCHITECTURE = "lraspp"
SEG_CHECKPOINT = "pocs/mobilenet-lraspp-speed/model-lraspp-bn-satellite.pt"
SEG_CHANNELS = "all"
MATCH_GSD_M = 0.385
FRAMES_DATASET = "village-crossdate-frames"

GATE = "full"                                        # 007.8 / flight simulator gated setup
GATE_S_MAX_M = 3.0
GATE_C_MIN = 0.01
GATE_TEMPERATURE = 0.25

# Data

In [ ]:
set_seed(SEED)
OUT = REPO_ROOT / "builds" / BUILD
RUN_CONFIG = {
    "method": "segmentation", "matching": MATCHING, "class_weights": CLASS_WEIGHTS,
    "noise_m": NOISE_M, "primary_noise_m": PRIMARY_NOISE_M, "n_directions": N_DIRECTIONS,
    "window": WINDOW, "search_scale": SEARCH_SCALE, "search_radius_m": None, "prior_sigma_m": None,
    "gate": "none", "fov_w_deg": FOV_W_DEG,
    "sat_feature_crop_pct": SAT_FEATURE_CROP_PCT, "drone_feature_crop_pct": DRONE_FEATURE_CROP_PCT,
    "seg_architecture": SEG_ARCHITECTURE, "seg_checkpoint_sat": SEG_CHECKPOINT,
    "seg_checkpoint_drone": SEG_CHECKPOINT, "seg_channels": SEG_CHANNELS,
    "drone_seg_gsd_m": MATCH_GSD_M, "match_gsd_m": MATCH_GSD_M,
    "frames_dataset": FRAMES_DATASET, "frames_dataset_ref": dataset_ref(FRAMES_DATASET),
    "places": PLACES, "map_date": MAP_DATE,
}
RUN_CONFIGS = {"ungated": RUN_CONFIG,
               "gated": {**RUN_CONFIG, "gate": GATE, "gate_s_max_m": GATE_S_MAX_M,
                         "gate_c_min": GATE_C_MIN, "temperature": GATE_TEMPERATURE}}
decision_configs = {rule: correlator_config(c) for rule, c in RUN_CONFIGS.items()}
decision_configs

In [ ]:
def frame_place(frame: DroneFrame) -> str:
    return json.loads((frame.image_path.parent / "meta.json").read_text())["place"]

In [ ]:
frames = [f for f in load_frames(FRAMES_DATASET) if frame_place(f) in PLACES]
places = {place: [f for f in frames if frame_place(f) == place] for place in PLACES}
maps = {place: load_date(place, MAP_DATE) for place in PLACES}
subsets = {"all": {f.name for f in frames}, **{p.split("-")[0]: {f.name for f in m} for p, m in places.items()}}
{place: len(members) for place, members in places.items()}

# Model

In [ ]:
model = load_segmenter(REPO_ROOT / SEG_CHECKPOINT, DEVICE, SEG_ARCHITECTURE)

# Run

In [ ]:
def identity_records(place: str) -> pd.DataFrame:
    """Every draw of one place with the satellite as its own query, scored under each decision rule."""
    features = SegmentationFeatures(model, model, maps[place], SEG_CHANNELS, MATCH_GSD_M, MATCH_GSD_M, DEVICE)
    correlator = SatelliteAsDroneCorrelator(features, decision_configs["ungated"])
    draws = [d for noise_m in NOISE_M
             for d in benchmark_draws(correlator, places[place], noise_m, N_DIRECTIONS, SEED)]
    return pd.concat([score_draws(draws, config).assign(place=place, rule=rule)
                      for rule, config in decision_configs.items()], ignore_index=True)

In [ ]:
records = pd.concat([identity_records(place) for place in tqdm(PLACES, desc="places")], ignore_index=True)
records["harm"] = records["error_m"] > records["prior_error_m"] + HARM_MARGIN_M
records.to_csv(OUT / "runs" / "identity-records.csv", index=False)
len(records)

In [ ]:
def summary_row(group: pd.DataFrame) -> dict[str, float]:
    """Mean error per injected level and place, plus harm and decision shares at the primary level."""
    m = summarise(group, subsets, PRIMARY_NOISE_M)
    at = f"{PRIMARY_NOISE_M:g}"
    return {**{f"@{n:g} m": m[f"mean_position_error_m_at_{n:g}m"] for n in NOISE_M},
            **{f"{s} @{at}": m[f"mean_position_error_m_{s}"] for s in subsets if s != "all"},
            f"harm @{at}": m[f"harm_share_at_{at}m"],
            **{f"{d} @{at}": m[f"{d}_share_at_{at}m"] for d in ("accept", "across", "reject")}}

In [ ]:
summary = pd.DataFrame({rule: summary_row(group) for rule, group in records.groupby("rule")}).T.round(3)
summary.to_csv(OUT / "runs" / "identity-summary.csv")
summary

In [ ]:
def reach_table(group: pd.DataFrame) -> pd.DataFrame:
    """Error split by whether the truth was inside the search window at all."""
    return group.groupby(["noise_m", "reachable"])["error_m"].agg(["median", "mean", "max", "count"]).round(3)

In [ ]:
reach = reach_table(records[records["rule"] == "ungated"])
reach.to_csv(OUT / "runs" / "identity-by-reachable.csv")
reach

In [ ]:
(OUT / "runs" / "identity-config.json").write_text(json.dumps(
    {"configs": RUN_CONFIGS, "n_frames": len(frames), "n_draws_per_rule": len(records) // len(decision_configs)},
    indent=1))

In [ ]:
def blind_frames(records: pd.DataFrame) -> list[str]:
    """Frames the correlator never finds a peak in: no weighted class survives its presence/std gates."""
    peaks = records.groupby("frame")["peak_z"].max()
    return sorted(peaks[peaks == 0].index)

In [ ]:
BLIND = blind_frames(records)
content = records[~records["frame"].isin(BLIND)]
{"blind_frames": BLIND, "content_frames": content["frame"].nunique()}

In [ ]:
def floor_table(group: pd.DataFrame) -> pd.DataFrame:
    """The corrector's own floor: error when the truth is inside the window, per injected level."""
    return group.groupby("noise_m").apply(lambda d: pd.Series({
        "draws": len(d), "reachable_share": d["reachable"].mean(), "harm_share": d["harm"].mean(),
        "median_m": d["error_m"].median(), "mean_m": d["error_m"].mean(),
        "median_reachable_m": d.loc[d["reachable"], "error_m"].median(),
        "mean_reachable_m": d.loc[d["reachable"], "error_m"].mean(),
        "p90_reachable_m": d.loc[d["reachable"], "error_m"].quantile(0.9),
        "max_reachable_m": d.loc[d["reachable"], "error_m"].max(),
    }), include_groups=False).round(3)

In [ ]:
floor = pd.concat({rule: floor_table(group) for rule, group in content.groupby("rule")}, names=["rule"])
floor.to_csv(OUT / "runs" / "identity-floor.csv")
floor

# Plots

In [ ]:
def plot_cdf(group: pd.DataFrame, ax: plt.Axes, title: str) -> None:
    for noise_m, level in group.groupby("noise_m"):
        errors = np.sort(level["error_m"].to_numpy())
        line, = ax.plot(errors, np.linspace(0, 1, len(errors)), label=f"{noise_m:g} m injected")
        ax.axvline(noise_m, color=line.get_color(), ls=":", lw=1)
    ax.set_xscale("log")
    ax.set_xlabel("position error after correction, m"); ax.set_ylabel("share of draws")
    ax.set_title(title); ax.legend(); ax.grid(alpha=.3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, (rule, group) in zip(axes, records.groupby("rule")):
    plot_cdf(group, ax, f"satellite as its own query — {rule}")
fig.tight_layout()
fig.savefig(OUT / "figures" / "identity-cdf.png", dpi=150)